In [1]:
import sys
print("python version",sys.version)
import torch
print("torch cuda version",torch.version.cuda)  # should NOT be None
print("torch version", torch.__version__)
print(torch.cuda.is_available())  # should be True

python version 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:27:36) [GCC 11.2.0]
torch cuda version 12.1
torch version 2.3.0+cu121
True


In [2]:
# Install latest Ultralytics (with YOLOv12 support)
!pip install ultralytics>=8.3.0 -q


✅ Dataset in YOLO format (structure)

In [3]:
from ultralytics import YOLO

# Load pre-trained YOLOv11n model (Check if 'yolov11n.pt' exists. Otherwise use 'yolov8n.pt')
try:
    test_model = YOLO('yolo12n.pt')
    print("✅ YOLOv12 is available!")
except Exception as e:
    print(f"ℹ️ YOLOv12 will be downloaded during training")

✅ YOLOv12 is available!


In [4]:
# Verify dataset structure
import os

base_path = "/home/ratheesh/Trignova/Dataset/yolo_data_carplate"

train_imgs = len(os.listdir(f"{base_path}/train/images"))
train_labels = len(os.listdir(f"{base_path}/train/labels"))

val_imgs = len(os.listdir(f"{base_path}/valid/images"))
val_labels = len(os.listdir(f"{base_path}/valid/labels"))

test_imgs = len(os.listdir(f"{base_path}/test/images"))
test_labels = len(os.listdir(f"{base_path}/test/labels"))

print("📊 Dataset Summary:")
print(f"   Train: {train_imgs} images, {train_labels} labels")
print(f"   Valid: {val_imgs} images, {val_labels} labels")
print(f"   Test:  {test_imgs} images, {test_labels} labels")
print(f"   Total: {train_imgs + val_imgs + test_imgs} images")

if train_imgs != train_labels or val_imgs != val_labels or test_imgs != test_labels:
    print("\n⚠️ Warning: Image/label count mismatch!")
else:
    print("\n✅ Dataset verified - ready for YOLOv12 training")



📊 Dataset Summary:
   Train: 10035 images, 10035 labels
   Valid: 2500 images, 2500 labels
   Test:  359 images, 359 labels
   Total: 12894 images

✅ Dataset verified - ready for YOLOv12 training


# Train YOLOv12 Model

In [10]:
# Load pre-trained YOLOv12n model
model = YOLO('yolo12n.pt')  # Latest YOLOv12 nano model
project='custom_yolo_train',  # Output folder
name='yolov12n_finetuned',    # Run name
path_yaml = base_path + '/data.yaml'
type(path_yaml),path_yaml

(str, '/home/ratheesh/Trignova/Dataset/yolo_data_carplate/data.yaml')

In [13]:

results = model.train(
    data= '/home/ratheesh/Trignova/Dataset/yolo_data_carplate/data.yaml',  # dataset config
    epochs=120,             # Enough for convergence
    batch=16,               # Adjust based on GPU memory
    imgsz=640,              # Standard YOLO input size
    device=0,               # Use GPU
    workers=8,              # Parallel data loading
    project='custom_yolo_train',
    name='yolov12n_finetuned',

    # Fine-tuning
    pretrained=True,        # start from pretrained weights
    patience=20,            # early stopping

    # Saving and logging
    save=True,
    save_period=10,         # save every 10 epochs
    plots=True,
    exist_ok=True,

    # Light augmentations for real-world robustness
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,             # horizontal flip with 50% chance
    mosaic=1.0,             # enable mosaic augmentation
    close_mosaic=10,        # disable mosaic in final epochs for stability

    # Optimization
    optimizer="auto",       # YOLO auto-selects the best optimizer
)

New https://pypi.org/project/ultralytics/8.3.220 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.155 🚀 Python-3.12.7 torch-2.3.0+cu121 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ratheesh/Trignova/Dataset/yolo_data_carplate/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name

100%|██████████████████████████████████████| 5.35M/5.35M [00:01<00:00, 3.62MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6409.6±2287.2 MB/s, size: 155.5 KB)


/home/ratheesh/anaconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,
train: Scanning /home/ratheesh/Trignova/Dataset/yolo_data_carplate/train/labels.

train: /home/ratheesh/Trignova/Dataset/yolo_data_carplate/train/images/I-27277_jpg.rf.5f15fd30334abe9f576fe4fc927cfa5f.jpg: 1 duplicate labels removed
train: /home/ratheesh/Trignova/Dataset/yolo_data_carplate/train/images/WhatsApp-Image-2022-01-28-at-9-18-58-AM_jpeg.rf.685229581c7fe6577ff9f68dd79492dc.jpg: 1 duplicate labels removed


...

val: New cache created: /home/ratheesh/Trignova/Dataset/yolo_data_carplate/valid/labels.cache
Plotting labels to custom_yolo_train/yolov12n_finetuned/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 113 weight(decay=0.0), 120 weight(decay=0.0005), 119 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to custom_yolo_train/yolov12n_finetuned
Starting training for 120 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/628 [00:00<?, ?it/s]/home/ratheesh/anaconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,
      1/120      3.43G      1.533      3.263     0.9931         57        640: 1
                 Class     Images  Instances      Box(P          R      mAP50  m


                   all       2500      21385      0.575      0.114     0.0713     0.0446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      2/120      3.57G      1.396      2.133     0.9299        199        640:  


KeyboardInterrupt: 

#  Validate Model 

In [ ]:
path_fine_tuned_model = 'custom_yolo_train/yolov12n_finetuned6/weights/best.pt'
path_test_image = "/home/ratheesh/Trignova/test_data_carplate"

In [ ]:
best_model = YOLO(path_fine_tuned_model)

print("✅ Model loaded ")

# Validate on test set
metrics = best_model.val()

print("\n📊 YOLOv12 Validation Metrics:")
print(f"   mAP@0.5:      {metrics.box.map50:.3f}  (Target: >0.85)")
print(f"   mAP@0.5:0.95: {metrics.box.map:.3f}   (Target: >0.70)")
print(f"   Precision:    {metrics.box.mp:.3f}     (Target: >0.80)")
print(f"   Recall:       {metrics.box.mr:.3f}     (Target: >0.80)")

if metrics.box.map50 > 0.87:
    print("\n✅ YOLOv12 performance: EXCELLENT (Better than YOLOv8!)")
elif metrics.box.map50 > 0.85:
    print("\n✅ YOLOv12 performance: EXCELLENT")
elif metrics.box.map50 > 0.70:
    print("\n✅ YOLOv12 performance: GOOD")
else:
    print("\n⚠️  Model performance: Acceptable (character detection can improve with fine-tuning)")

## Step 8: Test Inference

In [ ]:
model = YOLO(path_fine_tuned_model)
image1 = path_test_image + "1.jpg"
results = model(image1)
results[0].show()

# Export to Core ML (iOS)

In [ ]:
print("📱 Exporting YOLOv12 to Core ML (iOS)...")

# Export YOLOv12 to Core ML
export_path = best_model.export(
    format='coreml',
    imgsz=640,
    nms=True,
    half=False  # FP32 for compatibility
)

print("\n✅ YOLOv12 Core ML export complete!")


# Export to TensorFlow Lite (Android)

In [ ]:
print("🤖 Exporting YOLOv12 to TensorFlow Lite (Android)...")

# Export YOLOv12 to TFLite FP16
export_path = best_model.export(
    format='tflite',
    imgsz=640,
    int8=False,  # Use FP16
    nms=True
)

print("\n✅ YOLOv12 TFLite export complete!")